<img src="./static/imo_health.png" alt="IMO Health Logo" width="300"/>

---

# IMO Knowledge Graph Traversal

This notebook demonstrates querying the IMO Knowledge Graph GraphQL endpoint and formatting mappings in a table.

Endpoint: `https://api.imohealth.com/knowledgegraph/graphql/`

## Step 1: Install Packages and Load Configuration

Copy `config.json.template` to `config.json` and provide IMO client credentials.

In [ ]:
%pip install requests pandas --quiet

import json
import pathlib
import requests
import pandas as pd
from IPython.display import display

candidates = [
    pathlib.Path('config.json'),
    pathlib.Path.cwd() / 'config.json',
    pathlib.Path.cwd().parent / 'config.json'
]
cfg_path = next((p for p in candidates if p.exists()), None)

if cfg_path is None:
    raise FileNotFoundError('config.json not found. Copy config.json.template to config.json and fill in credentials.')

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

kg_cfg = cfg.get('knowledge_graph', {})
CLIENT_ID = kg_cfg.get('client_id', '')
CLIENT_SECRET = kg_cfg.get('client_secret', '')
TOKEN_URL = kg_cfg.get('token_url', 'https://api.imohealth.com/oauth/token')
GRAPHQL_URL = kg_cfg.get('graphql_url', 'https://api.imohealth.com/knowledgegraph/graphql/')

print(f'Loaded config from: {cfg_path.resolve()}')
print('client_id configured:', bool(CLIENT_ID and CLIENT_ID != '<YOUR_KNOWLEDGE_GRAPH_CLIENT_ID>'))
print('client_secret configured:', bool(CLIENT_SECRET and CLIENT_SECRET != '<YOUR_KNOWLEDGE_GRAPH_CLIENT_SECRET>'))

## Step 2: Get OAuth Access Token

In [ ]:
def get_token(client_id: str, client_secret: str) -> str:
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'audience': 'https://api.imohealth.com'
    }
    resp = requests.post(TOKEN_URL, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()['access_token']

access_token = get_token(CLIENT_ID, CLIENT_SECRET)
print('Token acquired (prefix):', access_token[:20] + '...')

## Step 3: Query the Mappings in Knowledge Graph 

First query: retrieve code system mappings for IMO lexical code `85191` 

In [ ]:
graphql_query = '''
query get_mappings{
  lexical(code: "85191") {
    title
    mappings{
      code
      codeSystem
    }
  }
}
'''

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/json'
}

response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': graphql_query},
    timeout=60
)
response.raise_for_status()
result = response.json()

lexical = result.get('data', {}).get('lexical', {})
title = lexical.get('title', '')
mappings = lexical.get('mappings', [])

print('Lexical title:', title)
df = pd.DataFrame(mappings)

styled_df = df.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
    {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '60%')]} 
])
display(styled_df)

print('Raw JSON response:')
print(json.dumps(result, indent=2))

## Step 4: Navigate Concept Hierarchy

Run a hierarchy query for lexical code `85191` and display broader/narrower concepts in pretty tables.

In [ ]:
hierarchy_query = '''
query get_hierachy{
  lexical(code: "85191") {
    title
    broader {
      code
      title
      mappings {
        code
        codeSystem
      }
    }
    narrower {
      code
      title
    }
  }
}
'''

hierarchy_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': hierarchy_query},
    timeout=60
)
hierarchy_response.raise_for_status()
hierarchy_result = hierarchy_response.json()

hier_lexical = hierarchy_result.get('data', {}).get('lexical', {})
hier_title = hier_lexical.get('title', '')
broader = hier_lexical.get('broader', [])
narrower = hier_lexical.get('narrower', [])

print('Lexical title:', hier_title)

def purple_style(df: pd.DataFrame, width: str = '80%'):
    return df.style.set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
        {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
        {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', width)]}
    ])

# Broader concepts table
broader_df = pd.DataFrame([{'code': b.get('code', ''), 'title': b.get('title', '')} for b in broader])
print('\nBroader Concepts')
display(purple_style(broader_df, width='55%'))

# Broader mappings table (flattened)
broader_mappings_rows = []
for b in broader:
    for m in b.get('mappings', []):
        broader_mappings_rows.append({
            'broader_code': b.get('code', ''),
            'broader_title': b.get('title', ''),
            'mapping_code': m.get('code', ''),
            'code_system': m.get('codeSystem', '')
        })

broader_mappings_df = pd.DataFrame(broader_mappings_rows)
print('\nBroader Mappings')
display(purple_style(broader_mappings_df, width='100%'))

# Narrower concepts table
narrower_df = pd.DataFrame([{'code': n.get('code', ''), 'title': n.get('title', '')} for n in narrower])
print('\nNarrower Concepts')
display(purple_style(narrower_df, width='100%'))

print('\nRaw JSON response:')
print(json.dumps(hierarchy_result, indent=2))

## Step 5: Drill Down Narrower Relationships

Run a second-level hierarchy query and show parent-to-child narrower relationships in a pretty table.

In [ ]:
drill_down_query = '''
query get_drill_down_hierarchy{
  lexical(code: "85191") {
    title
    narrower {
      code
      title
      narrower {
        code
        title
      }
    }
  }
}
'''

drill_down_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': drill_down_query},
    timeout=60
)
drill_down_response.raise_for_status()
drill_down_result = drill_down_response.json()

drill_lexical = drill_down_result.get('data', {}).get('lexical', {})
drill_title = drill_lexical.get('title', '')
level1_narrower = drill_lexical.get('narrower', [])

print('Lexical title:', drill_title)

# Reuse existing purple table style helper if available.
if 'purple_style' in globals():
    style_fn = purple_style
else:
    def style_fn(df: pd.DataFrame, width: str = '100%'):
        return df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', width)]}
        ])

# Level 1 narrower summary
level1_df = pd.DataFrame([
    {
        'level1_code': item.get('code', ''),
        'level1_title': item.get('title', ''),
        'child_count': len(item.get('narrower', []))
    }
    for item in level1_narrower
])
print('\nLevel 1 Narrower Concepts (with child counts)')
display(style_fn(level1_df, width='100%'))

# Flatten level1 -> level2 relationships
drill_rows = []
for parent in level1_narrower:
    parent_code = parent.get('code', '')
    parent_title = parent.get('title', '')
    children = parent.get('narrower', [])

    if children:
        for child in children:
            drill_rows.append({
                'parent_code': parent_code,
                'parent_title': parent_title,
                'child_code': child.get('code', ''),
                'child_title': child.get('title', '')
            })
    else:
        drill_rows.append({
            'parent_code': parent_code,
            'parent_title': parent_title,
            'child_code': '',
            'child_title': ''
        })

drill_df = pd.DataFrame(drill_rows)
print('\nDrill-Down Narrower Relationships (Level 1 -> Level 2)')
display(style_fn(drill_df, width='100%'))

print('\nRaw JSON response:')
print(json.dumps(drill_down_result, indent=2))

## Step 6: Query Allowed Refinements and Groups

Run a refinements query for lexical code 85191 and show each concept with its IMO Health refinement group.

Use the Knowledge graph content on Concept Refinements to build Refinement workflows.

In [ ]:
refinements_query = '''
query get_refinements{
  lexical(code: "85191") {
    title
    allowedRefinements{
      code
      title
      group {
        code
        title
      }
    }
  }
}
'''

refinements_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': refinements_query},
    timeout=60
)
refinements_response.raise_for_status()
refinements_result = refinements_response.json()

ref_lexical = refinements_result.get('data', {}).get('lexical', {})
ref_title = ref_lexical.get('title', '')
allowed_refinements = ref_lexical.get('allowedRefinements', [])

print('Lexical title:', ref_title)

refinement_rows = []
for r in allowed_refinements:
    g = r.get('group', {})
    refinement_rows.append({
        'concept_code': r.get('code', ''),
        'concept_title': r.get('title', ''),
        'group_code': g.get('code', ''),
        'group_title': g.get('title', '')
    })

refinements_df = pd.DataFrame(refinement_rows)

if 'purple_style' in globals():
    style_fn = purple_style
else:
    def style_fn(df: pd.DataFrame, width: str = '100%'):
        return df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', width)]}
        ])

print('\nRefinement Concepts with Groups')
display(style_fn(refinements_df, width='100%'))

group_summary_df = (
    refinements_df.groupby(['group_code', 'group_title'], dropna=False)
    .size()
    .reset_index(name='concept_count')
    .sort_values(by='concept_count', ascending=False)
    .reset_index(drop=True)
)

print('\nRefinement Group Summary')
display(style_fn(group_summary_df, width='70%'))

print('\nRaw JSON response:')
print(json.dumps(refinements_result, indent=2))

## Step 7: Filter Narrower Concepts by Refinement

Run a query that filters `narrower` concepts by a selected refinement code (`767164103`) from `allowedRefinements`. This is how user selected refinements from the allowedRefinements can be applied in a workflow

In [ ]:
filtered_refinements_query = '''
query get_refinements{
  lexical(code: "85191") {
    title
    narrower(refinements: ["767164103"]){
      code
      title
    }
    allowedRefinements{
      code
      title
      group {
        code
        title
      }
    }
  }
}
'''

filtered_refinements_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': filtered_refinements_query},
    timeout=60
)
filtered_refinements_response.raise_for_status()
filtered_refinements_result = filtered_refinements_response.json()

filtered_lexical = filtered_refinements_result.get('data', {}).get('lexical', {})
filtered_title = filtered_lexical.get('title', '')
filtered_narrower = filtered_lexical.get('narrower', [])
filtered_allowed = filtered_lexical.get('allowedRefinements', [])

print('Lexical title:', filtered_title)

filtered_narrower_df = pd.DataFrame([
    {'code': n.get('code', ''), 'title': n.get('title', '')}
    for n in filtered_narrower
])
filtered_allowed_df = pd.DataFrame([
    {
        'refinement_code': a.get('code', ''),
        'refinement_title': a.get('title', ''),
        'group_code': a.get('group', {}).get('code', ''),
        'group_title': a.get('group', {}).get('title', '')
    }
    for a in filtered_allowed
])

if 'purple_style' in globals():
    style_fn = purple_style
else:
    def style_fn(df: pd.DataFrame, width: str = '100%'):
        return df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', width)]}
        ])

print('\nFiltered Narrower Concepts (refinement code: 767164103)')
display(style_fn(filtered_narrower_df, width='60%'))

print('\nAllowed Refinements')
display(style_fn(filtered_allowed_df, width='100%'))

print('\nRaw JSON response:')
print(json.dumps(filtered_refinements_result, indent=2))

## Step 8: Cross Domain Relationships

The content is coming soon to the graph. Stay tuned for queries.